# Cortex Multi-Dataset GPU Pipeline

Notebook Google Colab complet pour Cortex.

Le pipeline produit des **risk signals** pour Cortex. Il ne prend jamais la decision finale. Les actions finales sont seulement simulees.

## 1. Setup & GPU check

In [ ]:
import os
import sys
from pathlib import Path

ROOT_CANDIDATES = [Path.cwd(), Path('/content/coco'), Path('/content/drive/MyDrive/coco')]
REPO_ROOT = None
for candidate in ROOT_CANDIDATES:
    if (candidate / 'scripts' / 'runtime' / 'cortex_colab_multidataset_pipeline.py').exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise FileNotFoundError('Repo Cortex introuvable. Monter le repo dans /content/coco ou ouvrir ce notebook depuis le depot.')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('REPO_ROOT =', REPO_ROOT)

## 2. Installation packages

In [ ]:
%pip -q install --disable-pip-version-check --no-warn-script-location pandas numpy scikit-learn matplotlib seaborn networkx plotly tqdm pyarrow fastparquet ipywidgets jupyterlab_widgets widgetsnbextension
%pip -q install --disable-pip-version-check --no-warn-script-location torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip -q install --disable-pip-version-check --no-warn-script-location torch-geometric

## Jupyter AI integration

Ce bloc permet d'utiliser `jupyter-ai` comme copilote d'analyse dans le notebook.

Utiliser une variable d'environnement standard comme `OPENAI_API_KEY`. Ne jamais stocker la cle en dur dans le notebook.

In [ ]:
import sys

JUPYTER_AI_SUPPORTED = sys.version_info < (3, 14)
if JUPYTER_AI_SUPPORTED:
    get_ipython().run_line_magic('pip', 'install -q --disable-pip-version-check --no-warn-script-location jupyter-ai')
else:
    print('Jupyter AI install skipped: Python 3.14+ is not reliably supported by current langchain/pydantic stack.')

In [ ]:
import os
import getpass

if JUPYTER_AI_SUPPORTED:
    if not os.environ.get('OPENAI_API_KEY'):
        os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter your OpenAI API key: ')
    masked = os.environ['OPENAI_API_KEY'][:7] + '...' if os.environ.get('OPENAI_API_KEY') else 'missing'
    print('OPENAI_API_KEY set:', masked)
else:
    print('Jupyter AI key prompt skipped: runtime is Python 3.14+.')

In [ ]:
if JUPYTER_AI_SUPPORTED:
    try:
        get_ipython().run_line_magic('load_ext', 'jupyter_ai_magics')
        print('jupyter_ai_magics loaded')
    except Exception as exc:
        print('Jupyter AI magics not loaded:', exc)
        print('In Colab, this can depend on the current frontend/runtime integration.')
else:
    print('Jupyter AI magics disabled on Python 3.14+ for notebook stability.')

In [ ]:
JUPYTER_AI_PIPELINE_BRIEF = '''
You are the notebook copilot for the Cortex multi-dataset GPU pipeline.
Do not make final security decisions.
Do not fabricate missing datasets, metrics, or model quality.
Use only notebook variables and outputs already computed.

Your tasks from A to Z are:
A. Inspect dataset_map and raw_df and summarize source coverage.
B. Review DataTrustScore behavior and quantify accepted, quarantined, rejected records.
C. Explain whether filtering is too strict or too permissive.
D. Review agent_datasets and explain what each agent receives.
E. Analyze engineered features: temporal_drift, graph_expansion, novelty, baseline_deviation, insider_score, admin_anomaly, edge_suspicion.
F. Inspect graph_bundle and identify the highest-risk entities.
G. Review training loss curves for autoencoder, temporal model, and graph model.
H. Explain anomaly_score, novelty_score, trust_score, temporal_score, graph_score.
I. Explain reasoning_score and detection_score composition.
J. Audit orchestrated_df for high-risk and critical-risk cases.
K. Review agent_messages and identify consensus and disagreement.
L. Analyze stream_df and continuous_df for realtime drift and model stability.
M. Summarize validation metrics with strengths and weaknesses.
N. Separate direct observations from inferences.
O. Suggest the next notebook cells to run if something is missing.
P. End with a section named: Cortex risk signals to review.
'''

print(JUPYTER_AI_PIPELINE_BRIEF)

In [ ]:
JUPYTER_AI_PIPELINE_PROMPT = (
    JUPYTER_AI_PIPELINE_BRIEF
    + "\nAnalyze `trusted_df`, `agent_datasets`, `loss_report`, `orchestrated_df`, `continuous_df`, and `metrics`."
)
print(JUPYTER_AI_PIPELINE_PROMPT)

Exemple d'utilisation avec Jupyter AI une fois les magics charges:

```python
%%ai openai:gpt-4.1-mini
{JUPYTER_AI_PIPELINE_BRIEF}
Analyze `trusted_df`, `agent_datasets`, `loss_report`, `orchestrated_df`, `continuous_df`, and `metrics`.
```

In [ ]:
import torch
from scripts.runtime.cortex_colab_multidataset_pipeline import detect_device, set_seed

set_seed(42)
device_info = detect_device()
device = torch.device(device_info['device'])
print('GPU name:', device_info['gpu_name'])
print('VRAM (GB):', device_info['vram_gb'])
print('Device used:', device)

## 3. Telechargement datasets

In [ ]:
from scripts.runtime.cortex_colab_multidataset_pipeline import download_or_generate_datasets

# Renseigner des URLs directes si vous avez des subsets exploitables.
DATASET_URLS = {
    # 'CIC-IDS2017': 'https://.../cic_ids2017_subset.csv',
    # 'UNSW-NB15': 'https://.../unsw_nb15_subset.csv',
}

data_dir = REPO_ROOT / 'test-artifacts' / 'colab-multidataset-cache'
dataset_map = download_or_generate_datasets(data_dir=data_dir, dataset_urls=DATASET_URLS, rows_per_dataset=3000)
{name: df.shape for name, df in dataset_map.items()}

## 4. Extraction & loading

In [ ]:
from scripts.runtime.cortex_colab_multidataset_pipeline import merge_datasets

raw_df = merge_datasets(dataset_map)
print('rows:', len(raw_df), 'columns:', len(raw_df.columns))
raw_df.head()

## 5. Data filtering & cleaning

In [ ]:
from scripts.runtime.cortex_colab_multidataset_pipeline import score_data_trust, filter_and_clean

trusted_df = score_data_trust(raw_df)
accepted_df, quarantine_df = filter_and_clean(trusted_df)
print('accepted:', len(accepted_df), 'quarantined_or_rejected:', len(quarantine_df))
trusted_df['data_trust_bucket'].value_counts()

## 6. Data selection par agent

In [ ]:
from scripts.runtime.cortex_colab_multidataset_pipeline import engineer_features, select_data_by_agent

features_df = engineer_features(accepted_df)
agent_datasets = select_data_by_agent(features_df)
{agent: frame.shape for agent, frame in agent_datasets.items()}

## 7. Feature engineering avance

In [ ]:
feature_cols = [
    'temporal_drift', 'graph_expansion', 'novelty', 'baseline_deviation',
    'insider_score', 'admin_anomaly', 'edge_suspicion', 'zero_day_proxy', 'data_trust_score'
]
features_df[feature_cols + ['label_attack', 'label_zero_day']].describe().T

## 8. Graph construction

In [ ]:
from scripts.runtime.cortex_colab_multidataset_pipeline import build_cortex_graph

graph_bundle = build_cortex_graph(features_df)
print('graph nodes:', graph_bundle.graph.number_of_nodes())
print('graph edges:', graph_bundle.graph.number_of_edges())
graph_bundle.node_frame.head()

## 9. Modeles ML

In [ ]:
from scripts.runtime.cortex_colab_multidataset_pipeline import train_all_models

models = train_all_models(features_df, graph_bundle, device=device)
{name: type(value).__name__ for name, value in models.items() if not name.endswith('history') and name != 'graph_scores'}

## 10. Entrainement GPU

In [ ]:
import pandas as pd

loss_report = pd.DataFrame({
    'autoencoder_loss': pd.Series(models['ae_history']),
    'temporal_loss': pd.Series(models['temporal_history']),
    'graph_loss': pd.Series(models['graph_history']),
})
loss_report.tail(10)

## 11. Scoring avance

In [ ]:
from scripts.runtime.cortex_colab_multidataset_pipeline import score_events

scored_df = score_events(features_df, graph_bundle, models, device=device)
scored_df[['anomaly_score', 'novelty_score', 'trust_score', 'temporal_score', 'graph_score', 'reasoning_score', 'detection_score']].describe().T

## 12. Orchestration agents

In [ ]:
from scripts.runtime.cortex_colab_multidataset_pipeline import orchestrate_agents

orchestrated_df, agent_messages = orchestrate_agents(scored_df)
print('agent messages:', len(agent_messages))
orchestrated_df[['event_id', 'primary_agent', 'risk_bucket', 'detection_score']].head()

In [ ]:
pd.DataFrame(agent_messages).head(15)

## 13. Visualisation

In [ ]:
from scripts.runtime.cortex_colab_multidataset_pipeline import plot_results

plot_results(orchestrated_df, graph_bundle)

## 14. Simulation temps reel

In [ ]:
from scripts.runtime.cortex_colab_multidataset_pipeline import simulate_realtime

stream_df = simulate_realtime(orchestrated_df, window=150)
stream_df[['timestamp', 'event_id', 'detection_score', 'rolling_risk', 'simulated_action']].tail(20)

## 15. Entrainement continu

In [ ]:
from scripts.runtime.cortex_colab_multidataset_pipeline import continuous_training_update

continuous_df = continuous_training_update(stream_df, replay_size=512)
continuous_df[['event_id', 'detection_score', 'continuous_detection_score']].tail(20)

## 16. Validation

In [ ]:
from scripts.runtime.cortex_colab_multidataset_pipeline import validate_pipeline

metrics = validate_pipeline(continuous_df)
pd.DataFrame([metrics]).T.rename(columns={0: 'value'})

## 17. Export

In [ ]:
from scripts.runtime.cortex_colab_multidataset_pipeline import export_pipeline_artifacts

export_dir = REPO_ROOT / 'test-artifacts' / 'colab-multidataset-pipeline'
exports = export_pipeline_artifacts(
    export_dir=export_dir,
    models=models,
    accepted_df=features_df,
    quarantine_df=quarantine_df,
    scored_df=continuous_df,
    agent_messages=agent_messages,
    metrics=metrics,
)
pd.DataFrame(list(exports.items()), columns=['artifact', 'path'])

## Resultat attendu

Le notebook montre:
- ingestion intelligente multi-datasets avec fallback synthetique
- filtrage robuste via DataTrustScore
- selection par agent
- entrainement GPU multi-modeles
- scoring avance et orchestration multi-agents
- simulation temps reel et apprentissage continu
- export des modeles, datasets filtres et resultats

Les sorties sont des **risk signals** et non des decisions finales.